# Atelier – Préparation de Données Textuelles

## Structure du projet

In [2]:
import os

# Racine du projet = dossier contenant data/ et notebooks/
PROJECT_ROOT = os.path.abspath(os.getcwd())
if not os.path.isdir(os.path.join(PROJECT_ROOT, "data")):
    PROJECT_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, ".."))

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
NOTEBOOKS_DIR = os.path.join(PROJECT_ROOT, "notebooks")

for d in [DATA_DIR, NOTEBOOKS_DIR]:
    os.makedirs(d, exist_ok=True)

Import des bibliothèques nécessaires et chargement du fichier `smart_reviews_raw.csv` depuis `data/`

In [3]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

RAW_PATH = os.path.join(DATA_DIR, "smart_reviews_raw.csv")
df = pd.read_csv(RAW_PATH)
df.head()

,id_avis,date,source,produit,texte,sentiment,note,langue
0,AV0001,2026-02-27,mobile,Ordinateur NovaBook,"Très bonne expérience, simple et efficace.",positif,4,fr
1,AV0002,2026-01-09,web,SmartPhone X,Très satisfait de mon achat 👍 #avis,positif,4,fr
2,AV0003,2026-07-03,réseaux_sociaux,Écouteurs AirSound,"Produit parfait, rien à signaler.",positif,4,fr
3,AV0004,2026-06-28,sav,SmartWatch Pro,"Produit excellent, je suis très satisfait. !!!",positif,5,fr
4,AV0005,2026-01-24,sav,SmartPhone X,LA BATTERIE TIENT VRAIMENT BIEN ET L'ÉCRAN EST SUPERBE.,positif,5,fr


## Partie 1 – Exploration du corpus

### 1) Charger les données CSV

In [4]:
print(f"Fichier chargé : {RAW_PATH}")
df.info()

Fichier chargé : /Users/fsarr/Documents/atelier_prepa_donnees_textuelles-/data/smart_reviews_raw.csv
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id_avis    1200 non-null   str  
 1   date       1200 non-null   str  
 2   source     1200 non-null   str  
 3   produit    1200 non-null   str  
 4   texte      1195 non-null   str  
 5   sentiment  1200 non-null   str  
 6   note       1200 non-null   int64
 7   langue     1200 non-null   str  
dtypes: int64(1), str(7)
memory usage: 75.1 KB


### 2)Combien d'avis contient le dataset ?

In [5]:
nb_avis = df.shape[0]
print(f"Nombre d'avis : {nb_avis}")


Nombre d'avis : 1200


### 3) Combien de colonnes possède-t-il ?

In [6]:
nb_colonnes = df.shape[1]
print(f"Nombre de colonnes : {nb_colonnes}")
print("Colonnes :", list(df.columns))

Nombre de colonnes : 8
Colonnes : ['id_avis', 'date', 'source', 'produit', 'texte', 'sentiment', 'note', 'langue']


### 4) Quel est le type de chaque colonne ?

In [7]:
df.dtypes

id_avis        str
date           str
source         str
produit        str
texte          str
sentiment      str
note         int64
langue         str
dtype: object

### 5) Existe-t-il des valeurs manquantes ?

In [8]:
na_counts = df.isna().sum()
print(na_counts)
print(f"\nTotal de valeurs manquantes : {na_counts.sum()}")

id_avis      0
date         0
source       0
produit      0
texte        5
sentiment    0
note         0
langue       0
dtype: int64

Total de valeurs manquantes : 5


### 6) Identifier quelques types de texte en affichant par exemple : texte normal ; texte vide ; texte contenant une URL ; texte contenant une mention ; texte contenant un hashtag ; texte contenant des emojis ; texte avec beaucoup de ponctuation ; texte en majuscules ; texte avec répétition de caractères.

In [9]:
def show_example(mask, label, n=1):
    exemples = df.loc[mask, "texte"].dropna()
    print(f"--- {label} ({mask.sum()} occurrence(s)) ---")
    if len(exemples) > 0:
        for t in exemples.head(n):
            print(repr(t))
    else:
        print("(aucun exemple trouvé)")
    print()

texte = df["texte"].fillna("")

show_example(texte.str.strip().eq("") | df["texte"].isna(), "Texte vide")
show_example(texte.str.contains(r"https?://|www\.", regex=True), "Texte avec URL")
show_example(texte.str.contains(r"@\w+", regex=True), "Texte avec mention")
show_example(texte.str.contains(r"#\w+", regex=True), "Texte avec hashtag")
show_example(texte.str.contains(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]", regex=True), "Texte avec emojis")
show_example(texte.str.count(r"[!?.,]{3,}").gt(0), "Texte avec beaucoup de ponctuation")
show_example(texte.str.len().gt(0) & texte.eq(texte.str.upper()) & texte.str.contains(r"[A-Za-zÀ-ÿ]"), "Texte en majuscules")
mask_repetition = texte.apply(lambda t: bool(re.search(r"(.)\1{2,}", t)))
show_example(mask_repetition, "Texte avec répétition de caractères")

# Un exemple de texte "normal" (aucun des cas ci-dessus)
mask_special = texte.str.contains(r"https?://|www\.|@\w+|#\w+|[\U0001F300-\U0001FAFF\u2600-\u27BF]", regex=True)
mask_normal = ~(mask_special | mask_repetition) & texte.str.len().gt(0)
show_example(mask_normal & texte.str.len().gt(0), "Texte normal")


--- Texte vide (6 occurrence(s)) ---
'   '

--- Texte avec URL (143 occurrence(s)) ---
'Produit parfait, rien à signaler. https://example.com/commande/17'

--- Texte avec mention (128 occurrence(s)) ---
'@client Livraison rapide et produit conforme à mes attentes.'

--- Texte avec hashtag (127 occurrence(s)) ---
'Très satisfait de mon achat 👍 #avis'

--- Texte avec emojis (192 occurrence(s)) ---
'Très satisfait de mon achat 👍 #avis'

--- Texte avec beaucoup de ponctuation (141 occurrence(s)) ---
'Produit excellent, je suis très satisfait. !!!'

--- Texte en majuscules (146 occurrence(s)) ---
"LA BATTERIE TIENT VRAIMENT BIEN ET L'ÉCRAN EST SUPERBE."

--- Texte avec répétition de caractères (165 occurrence(s)) ---
'Produit excellent, je suis très satisfait. !!!'

--- Texte normal (482 occurrence(s)) ---
'Très  bonne  expérience,  simple  et  efficace.'

